# Fruits Vector Demo — Stage 2: Similarity Metrics
With the LanceDB table from Stage 1 ready, we now load the stored fruit vectors, reissue the mango query, and compare cosine, Euclidean, and dot-product rankings to show how the neighbour order shifts while the winner remains stable.

## Reload the stored fruit vectors
This cell reopens the LanceDB path, reads every fruit record, and prepares the mango query so the notebook works independently without rerunning Stage 1 cells.

In [ ]:
import numpy as np
import lancedb
from pathlib import Path

DB_PATH = Path("fruits_lancedb")
TABLE_NAME = "fruit_vectors"

db = lancedb.connect(str(DB_PATH))
table = db.open_table(TABLE_NAME)
stored = table.to_pandas()
fruit_vectors = {row.name: np.array(row.vector) for row in stored.itertuples()}
mango_query = np.array([0.9, 0.34, 0.08])
print(f"Reloaded {len(fruit_vectors)} fruits from LanceDB.")

## Compare similarity metrics
We compute cosine similarity, Euclidean distance, and dot product for each fruit and print the rankings to show how the order changes across metrics while the mango neighbour stays consistent.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))

def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))

metrics = [
    ("Cosine similarity (desc)", sorted(
        ((name, cosine_similarity(vec, mango_query)) for name, vec in fruit_vectors.items()),
        key=lambda item: item[1],
        reverse=True
    )),
    ("Euclidean distance (asc)", sorted(
        ((name, euclidean_distance(vec, mango_query)) for name, vec in fruit_vectors.items()),
        key=lambda item: item[1]
    )),
    ("Dot product (desc)", sorted(
        ((name, dot_product(vec, mango_query)) for name, vec in fruit_vectors.items()),
        key=lambda item: item[1],
        reverse=True
    )),
]
for title, ranking in metrics:
    print(f"
{title}:")
    for rank, (name, value) in enumerate(ranking[:3], start=1):
        print(f"{rank}. {name} — {value:.4f}")